In [63]:
import random

class unoCard:
    def __init__(self, color, number, skip=False):
        self.color=color
        self.number=number
        self.skip=skip

    def __str__(self):
        if self.skip:
            return f"{self.color} Skipped"
        return f"{self.color} {self.number}" 
        
def deckGeneration():
    colors=["Red", "Blue", "Green", "Yellow"]
    deck=[]

    for c in colors:
        for n in range(10):
            deck.append(unoCard(c, n))
        deck.append(unoCard(c, None, skip=True))

    random.shuffle(deck)
    return deck

def cardDealing(deck):
    return [deck.pop() for x in range(5)] 

def validMoves(hand, topCard):
    valid=[]

    for card in hand:
        if card.color==topCard.color or card.number==topCard.number or card.skip:
            valid.append(card)

    return valid

class gameState:
    def __init__(self, p1, p2, p3, topCard, deck):
        self.p1=p1
        self.p2=p2
        self.p3=p3
        self.topCard=topCard
        self.deck=deck

def cardMovement(state, player, move):
    newState=gameState(state.p1[:], state.p2[:], state.p3[:], state.topCard,
                       state.deck[:])

    hand=getattr(newState, player)

    if move is None:
        if newState.deck:
            hand.append(newState.deck.pop())

        return newState

    hand.remove(move)
    newState.topCard=move

    return newState

def evaluationFunction_defensive(state):
    aiCards=len(state.p1)
    avgOpponent=(len(state.p2) + len(state.p3))/2
    skip=sum(1 for c in state.p1 if c.skip)
    
    score=50 - 5*aiCards + 4*avgOpponent + 5*skip
    return score

def evaluationFunction_offensive(state):
    aiCards=len(state.p2)
    avgOpponent=(len(state.p1) + len(state.p3))/2
    skip=sum(1 for c in state.p2 if c.skip)
    
    score=50 - 8*aiCards + avgOpponent + skip
    return score

def minimax(state, depth, maximizing, player):
    if depth==0:
        return evaluationFunction_defensive(state)

    hand=getattr(state, player)
    validMove=validMoves(hand, state.topCard)

    if not validMove:
        validMove=[None]

    if maximizing:
        best=float('-inf')
        for move in validMove:
            newState=cardMovement(state, player, move)
            val=minimax(newState, depth-1, False, "p2")
            best=max(best, val)
        return best
    else:
        best=float('inf')
        for move in validMove:
            newState=cardMovement(state, player, move)
            val=minimax(newState, depth-1, True, "p1")
            best=min(best, val)
        return best

def expectimax(state, depth, nodeType):
    if depth==0:
        return evaluationFunction_offensive(state)

    validMove1=validMoves(state.p1, state.topCard)
    validMove2=validMoves(state.p2, state.topCard)

    if not validMove1:
        validMove1=[None]
    if not validMove2:
        validMove2=[None]
        
    if nodeType=="max":
        best=float('-inf')
        for move in validMove2:
            newState=cardMovement(state, "p2", move)
            val=expectimax(newState, depth-1, "chance")
            best=max(best, val)
        
        return best
    
    elif nodeType=="chance":
        if not state.deck:
            return evaluationFunction(state, "p2")

        prob=1/len(state.deck)
        total=0

        for card in state.deck:
            newState=cardMovement(state, "p2", None)
            total+=prob * expectimax(newState, depth-1, "opponent")
        
        return total
    
    elif nodeType=="opponent":
        prob=1/len(validMove1)
        total=0
        
        for move in validMove1:
            newState=cardMovement(state, "p1", move)
            total+=prob * expectimax(newState, depth-1, "max")
        
        return total

In [66]:
def gameLoop():
    mode=input("Enter mode for Player 3 (manual/auto):").lower()
    deck=deckGeneration()

    p1=cardDealing(deck)
    p2=cardDealing(deck)
    p3=cardDealing(deck)

    topCard=deck.pop()

    skipNext=False

    state=gameState(p1, p2, p3, topCard, deck)

    turn=0

    while True:
        if skipNext:
            print(f"Player {turn+1} is skipped :(")
            turn=(turn+1)%3
            skipNext=False
            continue
        
        print("\nThe top card is:", state.topCard)

        if turn==0:
            print("Player 1 (MINIMAX)")
            moves=validMoves(state.p1, state.topCard)

            bestMove=None
            bestScore=float('-inf')

            for move in moves or [None]:
                newState=cardMovement(state, "p1", move)
                score=minimax(newState, 3, False, "p2")

                print(f"Move: {move}, Score: {score}")

                if score>bestScore:
                    bestScore=score
                    bestMove=move

            state=cardMovement(state, "p1", bestMove)
            if state.topCard.skip:
                skipNext=True

        elif turn==1:
            print("Player 2 (EXPECTIMAX)")
            moves=validMoves(state.p2, state.topCard)

            bestMove=None
            bestScore=float('-inf')

            for move in moves or [None]:
                newState=cardMovement(state, "p2", move)
                score=expectimax(newState, 3, "chance")

                print(f"Move: {move}, Score: {score}")

                if score>bestScore:
                    bestScore=score
                    bestMove=move

            state=cardMovement(state, "p2", bestMove)
            if state.topCard.skip:
                skipNext=True

        else:
            print("Player 3")
            moves=validMoves(state.p3, state.topCard) or [None]

            if mode=="manual":
                print("Your cards:")
                for x, card in enumerate(state.p3):
                    print(f"{x}: {card}")

                choice=int(input("Choose card number or press -1 to draw a card:"))
                if choice==-1:
                    move=None
                else:
                    move=state.p3[choice]

            else:
                moves=validMoves(state.p3, state.topCard)

                bestMove=None
                bestScore=float('-inf')
    
                for move in moves or [None]:
                    newState=cardMovement(state, "p3", move)
                    score=minimax(newState, 3, False, "p1")
    
                    print(f"Move: {move}, Score: {score}")
    
                    if score>bestScore:
                        bestScore=score
                        bestMove=move
            
            state=cardMovement(state, "p3", move)
            if state.topCard.skip:
                skipNext=True
            
        if len(state.p1)==0:
            print("\nPlayer 1 wins :)")
            break
        if len(state.p2)==0:
            print("\nPlayer 2 wins :)")
            break
        if len(state.p3)==0:
            print("\nPlayer 3 wins :)")
            break
 
        turn=(turn+1)%3    

gameLoop()

Enter mode for Player 3 (manual/auto): auto



The top card is: Red 2
Player 1 (MINIMAX)
Move: Red 8, Score: 41.0
Move: Yellow 2, Score: 41.0

The top card is: Red 8
Player 2 (EXPECTIMAX)
Move: Green Skipped, Score: 24.000000000000007
Move: Yellow Skipped, Score: 23.00000000000001
Move: Yellow 8, Score: 24.000000000000007
Player 3 is skipped :(

The top card is: Green Skipped
Player 1 (MINIMAX)
Move: None, Score: 44.0
Player 2 is skipped :(

The top card is: Green Skipped
Player 3
Move: Green 1, Score: 39.0

The top card is: Green 1
Player 1 (MINIMAX)
Move: Green 7, Score: 47.0

The top card is: Green 7
Player 2 (EXPECTIMAX)
Move: Yellow Skipped, Score: 29.499999999999982
Player 3 is skipped :(

The top card is: Yellow Skipped
Player 1 (MINIMAX)
Move: Yellow 3, Score: 50.0
Move: Yellow 5, Score: 50.0
Move: Yellow 2, Score: 50.0

The top card is: Yellow 3
Player 2 (EXPECTIMAX)
Move: Yellow 9, Score: 36.999999999999986
Move: Yellow 8, Score: 36.999999999999986
Move: Yellow 0, Score: 36.999999999999986

The top card is: Yellow 9
Play

STRATEGY EMPLOYED
- for player 1, it is a defensive algorithm. Its main focus is to defend against its opponnent's attacks which is the reason the score depends a lot on the skip cards since they aid in defense

- for player 2, it is an offensive algorithm. Its main focus is to finish its cards faster than the other players. It assumes the average outcome and not the worst-case leading it to take risks. 

BETTER ALGORITHM
- Expectimax performs better because it depends on probabilty of the cards rather than depending entirely on the worst case scenario as minimax does. Its evaluation function also applies big penalty for having more cards in its own hand, hence it focuses on losing cards faster.

EVALUATION FUNCTIONS
- score=50 - 5*aiCards + 4*avgOpponent + 5*skip (MINIMAX)
- Focuses on the importance of the opponents having more cards and its own hand having more skip cards for defensive play

- score=50 - 8*aiCards + avgOpponent + skip (EXPECTIMAX)
- Focuses on ensuring its own cards are less since -8*aiCards would lead to greater negative score which is not needed. It would push the algorithm to choose moves that are aggressive and result in faster wins. Its only focus is losing its own cards, so less weightage of opponent cards and skip cards. 

In [59]:
def printGameTree(state, depth, player, nodeType, indent=""):

    if depth == 0:
        print(indent + f"[Leaf] TopCard: {state.topCard}")
        return

    if player == "p1":
        hand = state.p1
    elif player == "p2":
        hand = state.p2
    else:
        hand = state.p3

    moves = validMoves(hand, state.topCard)

    if not moves:
        moves = [None]

    print(indent + f"{nodeType.upper()} NODE ({player}) - TopCard: {state.topCard}")

    for move in moves:
        moveLabel = str(move) if move else "DRAW"

        print(indent + f" ├── Move: {moveLabel}")

        newState = cardMovement(state, player, move)

        # Decide next player + node type
        if player == "p1":
            nextPlayer = "p2"
            nextType = "min"
        elif player == "p2":
            nextPlayer = "p1"
            nextType = "chance"
        else:
            nextPlayer = "p1"
            nextType = "max"

        printGameTree(newState, depth-1, nextPlayer, nextType, indent + " │   ")

deck = deckGeneration()

p1 = cardDealing(deck)
p2 = cardDealing(deck)
p3 = cardDealing(deck)

topCard = deck.pop()

state = gameState(p1, p2, p3, topCard, deck)

printGameTree(state, depth=2, player="p1", nodeType="max")

MAX NODE (p1) - TopCard: Yellow 4
 ├── Move: Yellow 2
 │   MIN NODE (p2) - TopCard: Yellow 2
 │    ├── Move: Green Skipped
 │    │   [Leaf] TopCard: Green Skipped
 │    ├── Move: Yellow 3
 │    │   [Leaf] TopCard: Yellow 3
 │    ├── Move: Yellow 9
 │    │   [Leaf] TopCard: Yellow 9
 ├── Move: Green 4
 │   MIN NODE (p2) - TopCard: Green 4
 │    ├── Move: Green Skipped
 │    │   [Leaf] TopCard: Green Skipped


In [62]:
def printExpectimaxTree(state, depth, nodeType, indent=""):

    if depth == 0:
        print(indent + f"[Leaf] TopCard: {state.topCard}")
        return

    print(indent + f"{nodeType.upper()} NODE - TopCard: {state.topCard}")
    
    if nodeType == "max":
        moves = validMoves(state.p2, state.topCard)
        if not moves:
            moves = [None]

        for move in moves:
            moveLabel = str(move) if move else "DRAW"
            print(indent + f" ├── Move: {moveLabel}")

            newState = cardMovement(state, "p2", move)

            printExpectimaxTree(newState, depth-1, "chance", indent + " │   ")

    elif nodeType == "chance":
        if not state.deck:
            print(indent + " └── No cards in deck")
            return

        print(indent + f" ├── Drawing from deck ({len(state.deck)} possibilities)")

        sampleDeck = state.deck[:3]   

        for card in sampleDeck:
            print(indent + f" │   ├── Possible Card: {card}")

            newState = cardMovement(state, "p2", None)

            printExpectimaxTree(newState, depth-1, "opponent", indent + " │   │   ")

    elif nodeType == "opponent":
        moves = validMoves(state.p1, state.topCard)
        if not moves:
            moves = [None]

        for move in moves:
            moveLabel = str(move) if move else "DRAW"
            print(indent + f" ├── Opponent Move: {moveLabel}")

            newState = cardMovement(state, "p1", move)

            printExpectimaxTree(newState, depth-1, "max", indent + " │   ")


deck = deckGeneration()

p1 = cardDealing(deck)
p2 = cardDealing(deck)
p3 = cardDealing(deck)

topCard = deck.pop()

state = gameState(p1, p2, p3, topCard, deck)

printExpectimaxTree(state, depth=2, nodeType="max")

MAX NODE - TopCard: Yellow 2
 ├── Move: Yellow Skipped
 │   CHANCE NODE - TopCard: Yellow Skipped
 │    ├── Drawing from deck (28 possibilities)
 │    │   ├── Possible Card: Green 0
 │    │   │   [Leaf] TopCard: Yellow Skipped
 │    │   ├── Possible Card: Green 8
 │    │   │   [Leaf] TopCard: Yellow Skipped
 │    │   ├── Possible Card: Blue 8
 │    │   │   [Leaf] TopCard: Yellow Skipped
